# ECM402: Lab 3, Part A - Neural Network from Scratch
Complete implementation including data generation, layer definitions, Adam optimizer, gradient checking, training, and decision boundary plotting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def generate_data(N=100, K=3, seed=42):
    np.random.seed(seed)
    D = 2
    X = np.zeros((N*K, D))
    y = np.zeros(N*K, dtype='uint8')
    for j in range(K):
        ix = range(N*j, N*(j+1))
        r = np.linspace(0.0, 1, N)
        theta = np.linspace(j*4, (j+1)*4, N) + np.random.randn(N)*0.05
        X[ix] = np.c_[r*np.sin(theta), r*np.cos(theta)]
        y[ix] = j
    return X, y

X, y = generate_data(N=100, K=3, seed=42)
y_one_hot = np.zeros((y.size, 3))
y_one_hot[np.arange(y.size), y] = 1
print("Dataset generated successfully.")

In [ ]:
class DenseLayer:
    def __init__(self, n_in, n_out):
        self.W = np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in)
        self.b = np.zeros((1, n_out))
        
    def forward(self, X):
        self.X = X
        return np.dot(X, self.W) + self.b
        
    def backward(self, dL_dz):
        self.dL_dW = np.dot(self.X.T, dL_dz)
        self.dL_db = np.sum(dL_dz, axis=0, keepdims=True)
        return np.dot(dL_dz, self.W.T)

class ReLU:
    def forward(self, z):
        self.z = z
        return np.maximum(0, z)
        
    def backward(self, dL_da):
        return dL_da * (self.z > 0)

class SoftmaxCrossEntropy:
    def forward(self, z, y_true):
        self.y_true = y_true
        shifted_z = z - np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(shifted_z)
        self.y_hat = exp_z / np.sum(exp_z, axis=1, keepdims=True)
        
        n = z.shape[0]
        y_hat_clipped = np.clip(self.y_hat, 1e-9, 1 - 1e-9)
        loss = -np.sum(y_true * np.log(y_hat_clipped)) / n
        return loss
        
    def backward(self):
        n = self.y_hat.shape[0]
        return (self.y_hat - self.y_true) / n

In [ ]:
class Adam:
    def __init__(self, layers, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.layers = layers
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0
        self.m = [{'W': np.zeros_like(l.W), 'b': np.zeros_like(l.b)} for l in layers]
        self.s = [{'W': np.zeros_like(l.W), 'b': np.zeros_like(l.b)} for l in layers]
        
    def update(self):
        self.t += 1
        for i, layer in enumerate(self.layers):
            for param, grad in [('W', layer.dL_dW), ('b', layer.dL_db)]:
                self.m[i][param] = self.beta1 * self.m[i][param] + (1 - self.beta1) * grad
                self.s[i][param] = self.beta2 * self.s[i][param] + (1 - self.beta2) * (grad ** 2)
                
                m_hat = self.m[i][param] / (1 - self.beta1 ** self.t)
                s_hat = self.s[i][param] / (1 - self.beta2 ** self.t)
                
                update = self.lr * m_hat / (np.sqrt(s_hat) + self.eps)
                if param == 'W':
                    layer.W -= update
                else:
                    layer.b -= update

In [ ]:
# Initialize a fresh, untrained network
layer1 = DenseLayer(2, 64)
relu1 = ReLU()
layer2 = DenseLayer(64, 3)
loss_fn = SoftmaxCrossEntropy()

def compute_loss(X, y_one_hot):
    z1 = layer1.forward(X)
    a1 = relu1.forward(z1)
    z2 = layer2.forward(a1)
    return loss_fn.forward(z2, y_one_hot)

def gradient_check(layer_to_check, param_name, X, y_one_hot, epsilon=1e-5):
    # Run a fresh pass to sync gradients
    z1 = layer1.forward(X)
    a1 = relu1.forward(z1)
    z2 = layer2.forward(a1)
    _ = loss_fn.forward(z2, y_one_hot)
    
    dz2 = loss_fn.backward()
    da1 = layer2.backward(dz2)
    dz1 = relu1.backward(da1)
    _ = layer1.backward(dz1)
    
    if param_name == 'W':
        analytic_grad = layer_to_check.dL_dW
        param = layer_to_check.W
    else:
        analytic_grad = layer_to_check.dL_db
        param = layer_to_check.b
        
    idx = tuple(np.random.randint(0, d) for d in param.shape)
    original_val = param[idx]
    
    param[idx] = original_val + epsilon
    loss_plus = compute_loss(X, y_one_hot)
    
    param[idx] = original_val - epsilon
    loss_minus = compute_loss(X, y_one_hot)
    
    param[idx] = original_val
    
    numeric_grad = (loss_plus - loss_minus) / (2 * epsilon)
    rel_error = abs(analytic_grad[idx] - numeric_grad) / (abs(analytic_grad[idx]) + abs(numeric_grad) + 1e-8)
    
    print(f"Checking {param_name} at {idx}:")
    print(f"  Analytic: {analytic_grad[idx]:.6e} | Numeric: {numeric_grad:.6e}")
    print(f"  Relative Error = {rel_error:.2e}")
    assert rel_error < 1e-5, "Gradient check failed!"

print("Running Gradient Checks...")
gradient_check(layer1, 'W', X, y_one_hot)
gradient_check(layer1, 'b', X, y_one_hot)

In [ ]:
network_layers = [layer1, layer2]
optimizer = Adam(network_layers, lr=0.01)
epochs = 2000

print("\nStarting training...")
for epoch in range(epochs):
    z1 = layer1.forward(X)
    a1 = relu1.forward(z1)
    z2 = layer2.forward(a1)
    loss = loss_fn.forward(z2, y_one_hot)
    
    dz2 = loss_fn.backward()
    da1 = layer2.backward(dz2)
    dz1 = relu1.backward(da1)
    _ = layer1.backward(dz1)
    
    optimizer.update()
    
    if epoch % 500 == 0 or epoch == epochs - 1:
        predictions = np.argmax(loss_fn.y_hat, axis=1)
        accuracy = np.mean(predictions == y)
        print(f"Epoch {epoch} | Loss: {loss:.4f} | Accuracy: {accuracy:.4f}")

def plot_decision_boundary(X, y, layer1, relu1, layer2):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))
    
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    
    z1 = layer1.forward(grid_points)
    a1 = relu1.forward(z1)
    z2 = layer2.forward(a1)
    Z = np.argmax(z2, axis=1)
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.8, cmap=plt.cm.Spectral)
    plt.scatter(X[:, 0], X[:, 1], c=y, s=40, cmap=plt.cm.Spectral, edgecolors='k')
    plt.title("Decision Boundary (64-Neuron Hidden Layer)")
    plt.show()

plot_decision_boundary(X, y, layer1, relu1, layer2)